<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных о горах

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas для набора данных о горах.

**Данные:**
- `data/mountains.csv` — информация о горах: название (mountain), метка (mountainLabel), координаты (coordinates), высота (elevation), тип горной породы (rockMaterialLabel)

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем `mountains.csv` в pandas DataFrame
3. Очищаем и переименовываем столбцы при необходимости
4. Смотрим структуру данных, проверяем типы и делаем быструю валидацию


## 🐱 [1] Клонируем репозиторий курса в Colab

In [23]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-AnastasiaKalyashova"  # ← изменено: имя вашего репозитория
repo_path = f"/content/{repo}"

if not os.path.exists(repo_path):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git  # ← изменено: URL вашего репозитория

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-AnastasiaKalyashova


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [24]:
# 🐱 Шаг 2A. Чтение mountains.csv с авто-поиском пути

import pandas as pd
import os

# Автоматически ищем файл
file_path = None
for root, dirs, files in os.walk("."):
    if "mountains.csv" in files:
        file_path = os.path.join(root, "mountains.csv")
        break

if not file_path:
    raise FileNotFoundError("❌ mountains.csv не найден! Проверьте, что файл добавлен в репозиторий или загрузите его вручную.")

# Читаем файл
df_mountains = pd.read_csv(file_path)

print(f"✅ Файл загружен из: {file_path}")
print(f"✅ Строк: {len(df_mountains)}")
print(f"✅ Столбцы: {list(df_mountains.columns)}")
print("\n📋 Первые 3 строки:")
display(df_mountains.head(3))

✅ Файл загружен из: ./data/mountains.csv
✅ Строк: 4390
✅ Столбцы: ['mountain', 'mountainLabel', 'coordinates', 'elevation', 'rockMaterialLabel']

📋 Первые 3 строки:


,mountain,mountainLabel,coordinates,elevation,rockMaterialLabel
0,http://www.wikidata.org/entity/Q513,Джомолунгма,Point(86.925 27.988055555),8848.86,горная порода
1,http://www.wikidata.org/entity/Q513,Джомолунгма,Point(86.925 27.988055555),8848.86,лёд
2,http://www.wikidata.org/entity/Q524,Везувий,Point(14.42919 40.82261),1281.00,Тефрит


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле из Викиданных есть **технические столбцы**, которые полезны для идентификации объектов, но мешают простому анализу:

- Столбец `mountain` содержит URL-ссылку на объект в Wikidata (например, `http://www.wikidata.org/entity/Q513`) — **сохраняем его для отладки**, но переименовываем в `URL`.
- Столбцы `mountainLabel` и `rockMaterialLabel` содержат читаемые названия (горы и типа породы).

В этом шаге мы:
- переименуем столбец с URL Wikidata (`mountain` → `URL`);
- переименуем `mountainLabel → mountain`, `rockMaterialLabel → rockMaterial`;
- приведём числовой столбец `elevation` (высота) к типу `float`.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0 (если потребуется).

> ⚠️ **Важно:** столбец `URL` пригодится, если нужно будет быстро перейти к оригинальной записи горы в Викиданных. Столбец `coordinates` остаётся без изменений — его парсинг (извлечение широты/долготы из формата `Point(x y)`) можно выполнить позже при визуализации.

In [25]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

if "mountainLabel" in df_mountains.columns:
 df_mountains = df_mountains.rename(columns={
 "mountain": "URL", # ← сохраняем, не удаляем
 "mountainLabel": "mountain",
 "rockMaterialLabel": "rockMaterial",
 })
 print("✅ Столбцы переименованы")
 print("Текущие столбцы:", list(df_mountains.columns))
else:
 print("⏭️ Уже переименовано, пропускаем")

print("\n✅ Данные готовы к анализу")
print("\nТекущие столбцы:", list(df_mountains.columns))
print("Тип данных в столбце elevation:", df_mountains["elevation"].dtype)

# Нормализуем регистр: всё в нижний + убираем пробелы по краям
print("До нормализации:", df_mountains['rockMaterial'].nunique(), "уникальных значений")
df_mountains['rockMaterial'] = (
 df_mountains['rockMaterial']
 .str.lower()
 .str.strip()
)
print("После нормализации:", df_mountains['rockMaterial'].nunique(), "уникальных значений")
print("\nТоп-10 пород после нормализации:")
print(df_mountains['rockMaterial'].value_counts().head(10))

✅ Столбцы переименованы
Текущие столбцы: ['URL', 'mountain', 'coordinates', 'elevation', 'rockMaterial']

✅ Данные готовы к анализу

Текущие столбцы: ['URL', 'mountain', 'coordinates', 'elevation', 'rockMaterial']
Тип данных в столбце elevation: float64
До нормализации: 108 уникальных значений
После нормализации: 108 уникальных значений

Топ-10 пород после нормализации:
rockMaterial
известняк                  834
песчаник                   518
гранит                     346
мергель                    301
конгломерат                289
lutite                     229
доломит                    173
андезит                    171
осадочная горная порода    161
базальт                    155
Name: count, dtype: int64


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о горах:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по высоте (`elevation`) — минимальная, максимальная, средняя высота и т.д.

Для удобства используем функцию `show_info(df, name)`, чтобы компактно вывести информацию о таблице.


In [29]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_mountains, "Горы (df_mountains)")

# 📈 Базовая статистика по высоте
print("\n📈 Статистика по высоте (elevation):")
print(df_mountains['elevation'].describe())


📊 Горы (df_mountains)
Размер: (4390, 5)
Столбцы: URL, mountain, coordinates, elevation, rockMaterial

Первые строки:
                                   URL     mountain  \
0  http://www.wikidata.org/entity/Q513  Джомолунгма   
1  http://www.wikidata.org/entity/Q513  Джомолунгма   
2  http://www.wikidata.org/entity/Q524      Везувий   
3  http://www.wikidata.org/entity/Q583      Монблан   
4  http://www.wikidata.org/entity/Q583      Монблан   

                  coordinates  elevation   rockMaterial  
0  Point(86.925 27.988055555)    8848.86  горная порода  
1  Point(86.925 27.988055555)    8848.86            лёд  
2    Point(14.42919 40.82261)    1281.00         тефрит  
3   Point(6.865 45.832777777)    4805.59         гранит  
4   Point(6.865 45.832777777)    4805.59          гнейс  

📈 Статистика по высоте (elevation):
count     4390.000000
mean      1581.164787
std       1300.251027
min        -39.000000
25%        692.000000
50%       1233.900000
75%       2283.500000
max      163

## ❄️ [4] Уникальный анализ: «многослойность» гор и «ледяной пояс» Земли

Ваши данные обладают редкой особенностью — **геологическая «многослойность»**: одна и та же гора может иметь несколько записей с разными типами пород и материалов. Например, Джомолунгма представлена двумя слоями: «горная порода» у основания и «лёд» на вершине.

В этом шаге мы исследуем два уникальных явления:

### 🗻 1. Геологическая «многослойность»
- Сколько **уникальных гор** скрыто в 4 390 записях?
- Какие горы имеют **наибольшее разнообразие материалов** (геологическая сложность)?
- Есть ли горы, представленные 5+ разными породами — признак сложной структуры?

### 🧊 2. «Ледяной пояс» Земли
Лёд не встречается равномерно — он концентрируется в определённом высотном диапазоне. Мы выявим:
- На какой **минимальной высоте** начинает формироваться постоянный ледник?
- В каком **диапазоне высот** лёд встречается чаще всего («ледяной пояс»)?
- Есть ли горы с льдом **ниже 4 000 м** — признак полярного климата?

> 💡 **Интересный факт**: В Гималаях ледники начинаются ~5 000 м, а в Антарктиде — уже на уровне моря. Анализ высоты появления льда поможет косвенно определить географическое положение гор!


In [34]:
# ❄️ Шаг 4. Анализ «многослойности» и «ледяного пояса»

print("=" * 70)
print("🏔️  ЧАСТЬ 1: Геологическая «многослойность» гор")
print("=" * 70)

# 1.1 Сравнение уникальных гор vs общего числа записей
total_records = len(df_mountains)
unique_mountains = df_mountains["mountain"].nunique()
print(f"\n📊 Всего записей: {total_records}")
print(f"📊 Уникальных гор (по названию): {unique_mountains}")
print(f"📊 Среднее число записей на гору: {total_records / unique_mountains:.2f}")
print(f"   → Это означает, что в среднем у каждой горы есть данные о {int(total_records / unique_mountains)} типах материалов!")

df_unique = (
 df_mountains
 .groupby('URL')
 .agg(
 mountain = ('mountain', 'first'), # название горы
 lon = ('lon', 'first'), # долгота (после парсинга)
 lat = ('lat', 'first'), # широта
 elevation = ('elevation', 'first'), # высота (одинакова для всех строк)
 rock_count = ('rockMaterial', 'nunique'), # число уникальных пород
 rocks = ('rockMaterial', list), # список всех пород
 )
 .reset_index()
)
print(f"✅ df_unique: {len(df_unique)} уникальных гор")
print(f" (было {len(df_mountains)} строк в длинном формате)")
print(f"\nСреднее число пород на гору: {df_unique['rock_count'].mean():.2f}")
print("\n📈 Корректная статистика по высоте (по уникальным горам):")
print(df_unique['elevation'].describe())

# 1.2 Распределение «многослойности»
layer_counts = df_mountains.groupby("mountain").size()
distribution = layer_counts.value_counts().sort_index()
print("\n📈 Распределение записей на гору:")
for layers, count in distribution.head(6).items():
    print(f"   {layers} слоя(ев): {count} гор(ы)")

# 1.3 Топ-10 гор с наибольшим разнообразием материалов
top_complex = (df_mountains.groupby("mountain")
               .agg({"rockMaterial": "nunique", "elevation": "first"})
               .sort_values("rockMaterial", ascending=False)
               .head(10))

# Физически невозможные значения (выше Джомолунгмы)
MAX_REAL = 8849 # высота Джомолунгмы в метрах
anomalies_high = df_mountains[df_mountains['elevation'] > MAX_REAL]
print(f"⚠️ Гор с elevation > {MAX_REAL} м: {len(anomalies_high)}")
if len(anomalies_high) > 0:
 print(anomalies_high[['URL', 'mountain', 'elevation', 'rockMaterial']].to_string())
# Отрицательные высоты тоже подозрительны
anomalies_neg = df_mountains[df_mountains['elevation'] < 0]
print(f"\n⚠️ Гор с отрицательной высотой: {len(anomalies_neg)}")
if len(anomalies_neg) > 0:
 print(anomalies_neg[['URL', 'mountain', 'elevation']].to_string())

print("\n🏆 Топ-10 гор с наибольшим разнообразием материалов:")
print(top_complex.reset_index().to_string(index=False))

print("\n" + "=" * 70)
print("🧊 ЧАСТЬ 2: «Ледяной пояс» Земли — где живёт лёд?")
print("=" * 70)

# 2.1 Поиск всех вариантов написания «лёд» (русский/английский/опечатки)
ice_names = {"лёд", "лед", "ice"}
df_mountains["is_ice"] = df_mountains["rockMaterial"].str.lower().isin(ice_names)

ice_records = df_mountains[df_mountains["is_ice"]]
total_ice = len(ice_records)
print(f"\n❄️  Записей с льдом: {total_ice} из {total_records} ({total_ice/total_records*100:.1f}%)")

if total_ice > 0:
    # 2.2 Минимальная и максимальная высота с льдом
    min_ice = ice_records["elevation"].min()
    max_ice = ice_records["elevation"].max()
    print(f"   Минимальная высота с льдом: {min_ice:.0f} м")
    print(f"   Максимальная высота с льдом: {max_ice:.0f} м")

    # 2.3 Высотные диапазоны для анализа «ледяного пояса»
    bins = [-100, 0, 2000, 4000, 5000, 6000, 7000, 8000, 10000]
    labels = ["< 0", "0–2 км", "2–4 км", "4–5 км", "5–6 км", "6–7 км", "7–8 км", "8+ км"]

    df_mountains["height_bin"] = pd.cut(df_mountains["elevation"], bins=bins, labels=labels, right=False)
    ice_by_bin = (df_mountains.groupby("height_bin")
                  .agg(total=("is_ice", "size"), ice=("is_ice", "sum"))
                  .assign(ice_pct=lambda x: (x["ice"] / x["total"] * 100).round(1))
                  .sort_index())

    print("\n📊 Распределение льда по высотным диапазонам:")
    print(ice_by_bin[["total", "ice", "ice_pct"]].rename(columns={
        "total": "Всего записей",
        "ice": "С льдом",
        "ice_pct": "% с льдом"
    }).to_string())

    # 2.4 Определение «ледяного пояса» — диапазон с максимальной концентрацией льда
    peak_bin = ice_by_bin["ice_pct"].idxmax()
    peak_value = ice_by_bin.loc[peak_bin, "ice_pct"]
    print(f"\n🎯 «Ледяной пояс» Земли: {peak_bin} — здесь лёд встречается в {peak_value}% записей!")

    # 2.5 Интересный факт: горы с льдом ниже 4000 м (полярные регионы?)
    low_ice = ice_records[ice_records["elevation"] < 4000]
    if len(low_ice) > 0:
        print(f"\n🔍 Необычные находки: {len(low_ice)} записей с льдом ниже 4000 м")
        print("   Примеры:")
        for _, row in low_ice.head(5).iterrows():
            print(f"   • {row['mountain']} ({row['elevation']:.0f} м) — {row['rockMaterial']}")
else:
    print("   ⚠️  Лёд не обнаружен в данных. Проверьте написание в столбце rockMaterial.")

# Очистка временных столбцов
df_mountains.drop(columns=["is_ice", "height_bin"], inplace=True, errors="ignore")

# Парсим WKT-координаты в два числовых столбца
df_mountains[['lon', 'lat']] = (
   df_mountains['coordinates']
 .str.extract(r'Point\(([^\s]+)\s+([^\s]+)\)')
 .astype(float)
)
print("✅ Координаты распарсены:")
print(df_mountains[['mountain', 'lon', 'lat']].head(3))
print(f"\nДиапазон широт: {df_mountains['lat'].min():.2f} — {df_mountains['lat'].max():.2f}")
print(f"Диапазон долгот: {df_mountains['lon'].min():.2f} — {df_mountains['lon'].max():.2f}")



🏔️  ЧАСТЬ 1: Геологическая «многослойность» гор

📊 Всего записей: 4390
📊 Уникальных гор (по названию): 2832
📊 Среднее число записей на гору: 1.55
   → Это означает, что в среднем у каждой горы есть данные о 1 типах материалов!
✅ df_unique: 2916 уникальных гор
 (было 4390 строк в длинном формате)

Среднее число пород на гору: 1.43

📈 Корректная статистика по высоте (по уникальным горам):
count     2916.000000
mean      1717.628226
std       1401.565946
min        -39.000000
25%        749.600000
50%       1409.500000
75%       2475.000000
max      16390.000000
Name: elevation, dtype: float64

📈 Распределение записей на гору:
   1 слоя(ев): 1814 гор(ы)
   2 слоя(ев): 638 гор(ы)
   3 слоя(ев): 308 гор(ы)
   4 слоя(ев): 35 гор(ы)
   5 слоя(ев): 14 гор(ы)
   6 слоя(ев): 13 гор(ы)
⚠️ Гор с elevation > 8849 м: 16
                                           URL                   mountain  elevation                   rockMaterial
133      http://www.wikidata.org/entity/Q59805               Маунт

/tmp/ipykernel_695/218048765.py:86: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ice_by_bin = (df_mountains.groupby("height_bin")


## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** гор и типов пород есть в данных;
- **какие горы самые высокие** (Топ‑5 по высоте);
- **какие типы пород встречаются чаще всего** (Топ‑10 по числу записей);
- **диапазон высот** гор в датасете.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_mountains["rockMaterialLabel"].value_counts().head(10)` даёт **Топ‑10 типов пород по частоте**.

Метод `.nlargest(n)` возвращает топ-N значений по указанному столбцу, поэтому
`df_mountains.nlargest(5, "elevation")` даёт **5 самых высоких гор**.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет: горы
print("\n📊 Датасет: Горы (df_mountains)")
print("Уникальных гор:", df_mountains["mountain"].nunique())
print("Уникальных типов пород/материалов:", df_mountains["rockMaterial"].nunique())

print("\n📈 Диапазон высот:")
print(f"Минимальная: {df_mountains['elevation'].min()} м")
print(f"Максимальная: {df_mountains['elevation'].max()} м")
print(f"Средняя: {df_mountains['elevation'].mean():.2f} м")

print("\n🏆 Топ-5 самых высоких гор:")
top_mountains = df_mountains.nlargest(5, "elevation")[["mountain", "elevation", "rockMaterial"]]
print(top_mountains.to_string(index=False))

print("\n📊 Топ-10 типов пород/материалов по частоте:")
print(df_mountains["rockMaterial"].value_counts().head(10))

print("\n📈 Распределение высот (квантили):")
print(df_mountains["elevation"].describe())

🔍 Быстрая проверка данных

📊 Датасет: Горы (df_mountains)
Уникальных гор: 2832
Уникальных типов пород/материалов: 108

📈 Диапазон высот:
Минимальная: -39.0 м
Максимальная: 16390.0 м
Средняя: 1581.16 м

🏆 Топ-5 самых высоких гор:
                 mountain  elevation rockMaterial
                 Блэкбёрн    16390.0       гранит
              Силл (гора)    14159.0       гранит
Рассел (гора, Калифорния)    14094.0       гранит
 Сплит (гора, Калифорния)    14064.0  Гранодиорит
Лэнгли (гора, Калифорния)    14032.0       гранит

📊 Топ-10 типов пород/материалов по частоте:
rockMaterial
известняк                  834
песчаник                   518
гранит                     346
Мергель                    301
Конгломерат                289
lutite                     229
доломит                    173
андезит                    171
осадочная горная порода    161
базальт                    155
Name: count, dtype: int64

📈 Распределение высот (квантили):
count     4390.000000
mean      1581.16478

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨